<a href="https://colab.research.google.com/github/rahulkumarjalandhara23-rgb/aiml-30-days/blob/main/day1_2_demo_sms_spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import glob
import pandas as pd

kaggle_files = glob.glob("/kaggle/input/**/spam.csv", recursive=True)

if kaggle_files:
    # Kaggle: the attached dataset. Its file needs latin-1 encoding and has 3 empty extra columns.
    raw = pd.read_csv(kaggle_files[0], encoding="latin-1")
    df = raw.iloc[:, :2]
    df.columns = ["label", "text"]
    print("Loaded from Kaggle:", kaggle_files[0])
else:
    # Colab or laptop: a public copy of the same UCI SMS Spam Collection
    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
    print("Loaded from the public copy on GitHub")

df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Messages (duplicates removed):", len(df))
df.sample(5, random_state=1)


Loaded from the public copy on GitHub
Messages (duplicates removed): 5169


,label,text
2971,ham,Ok. But i finish at 6.
3830,spam,87077: Kick off a new season with 2wks FREE go...
3066,ham,Ron say fri leh. N he said ding tai feng cant ...
3231,spam,Do you want a new Video phone? 600 anytime any...
2139,ham,"I prefer my free days... Tues, wed, fri oso ca..."


In [ ]:
counts = df["label"].value_counts()
print(counts)
print(f"\nSpam share: {counts['spam'] / len(df):.1%}")
print(f"A 'model' that always says HAM would already be {counts['ham'] / len(df):.1%} accurate, and catch zero spam.")

label
ham     4516
spam     653
Name: count, dtype: int64

Spam share: 12.6%
A 'model' that always says HAM would already be 87.4% accurate, and catch zero spam.


In [ ]:
SPAM_WORDS = ["free", "win", "prize", "claim", "urgent", "cash", "http", "www"]   # <- edit me

def rule_says_spam(text):
    t = text.lower()
    return any(word in t for word in SPAM_WORDS)

df["rule_pred"] = df["text"].apply(lambda t: "spam" if rule_says_spam(t) else "ham")

is_spam = df["label"] == "spam"
flagged = df["rule_pred"] == "spam"
print(f"Spam caught : {(is_spam & flagged).sum()} of {is_spam.sum()}")
print(f"Spam missed : {(is_spam & ~flagged).sum()}")
print(f"False alarms: {(~is_spam & flagged).sum()} genuine messages wrongly flagged")

Spam caught : 406 of 653
Spam missed : 247
False alarms: 138 genuine messages wrongly flagged


In [ ]:
pd.set_option("display.max_colwidth", 120)
print("GENUINE messages your rules flagged as spam:")
display(df[~is_spam & flagged][["text"]].head(5))
print("SPAM your rules missed:")
display(df[is_spam & ~flagged][["text"]].head(5))

GENUINE messages your rules flagged as spam:


,text
75,I am waiting machan. Call me once you free.
87,Yes I started to send requests to make it but pain came back so I'm back in bed. Double coins at the factory too. I ...
89,"Ela kano.,il download, come wen ur free.."
106,"Aight, I'll hit you up when I get some cash"
150,The wine is flowing and i'm i have nevering..


SPAM your rules missed:


,text
19,"England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SC..."
34,Thanks for your subscription to Ringtone UK your mobile will be charged £5/month Please confirm by replying YES or N...
68,"Did you hear about the new ""Divorce Barbie""? It comes with all of Ken's stuff!"
134,Want 2 get laid tonight? Want real Dogging locations sent direct 2 ur mob? Join the UK's largest Dogging Network bt ...
157,Customer service annoncement. You have a New Years delivery waiting for you. Please call 07046744435 now to arrange ...


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

model = make_pipeline(CountVectorizer(lowercase=True), MultinomialNB())
model.fit(X_train, y_train)
print("Trained on", len(X_train), "messages. Testing on", len(X_test), "it has never seen.")

Trained on 4135 messages. Testing on 1034 it has never seen.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

def report(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=["spam", "ham"])
    caught, missed, false_alarms = cm[0, 0], cm[0, 1], cm[1, 0]
    total_spam = caught + missed
    print(f"{name:<22} accuracy {accuracy_score(y_true, y_pred):6.1%} | "
          f"spam caught {caught}/{total_spam} | false alarms {false_alarms}")

rule_test_pred = ["spam" if rule_says_spam(t) else "ham" for t in X_test]
report("Always say ham", y_test, ["ham"] * len(y_test))
report("Your rules", y_test, rule_test_pred)
report("Naive Bayes (ML)", y_test, model.predict(X_test))

Always say ham         accuracy  87.3% | spam caught 0/131 | false alarms 0
Your rules             accuracy  92.3% | spam caught 77/131 | false alarms 26
Naive Bayes (ML)       accuracy  98.2% | spam caught 117/131 | false alarms 5


In [ ]:
import numpy as np

vec = model.named_steps["countvectorizer"]
nb = model.named_steps["multinomialnb"]
words = np.array(vec.get_feature_names_out())
spam_idx = list(nb.classes_).index("spam")
ham_idx = list(nb.classes_).index("ham")
spamminess = nb.feature_log_prob_[spam_idx] - nb.feature_log_prob_[ham_idx]

print("Top 15 words that push a message towards SPAM:")
print(", ".join(words[np.argsort(spamminess)[-15:][::-1]]))

Top 15 words that push a message towards SPAM:
claim, prize, 150p, tone, co, www, 18, 500, guaranteed, 1000, cs, uk, awarded, ringtone, 150ppm


In [ ]:
messages = [
    "Bhai, free ho toh call kar, notes chahiye for tomorrow's exam!",
    "Dear customer, your KYC is pending. Update now at kyc-update-bank.in or your account will be blocked today.",
    "Congrats! U hav w0n a 1Ph0ne. Cl!ck bit.ly/xyz",
    "Are we meeting at the canteen after the lab?",
    # add your own here
]

spam_col = list(model.classes_).index("spam")
for msg in messages:
    p_spam = model.predict_proba([msg])[0][spam_col]
    verdict = "SPAM" if p_spam >= 0.5 else "ham "
    rule = "spam" if rule_says_spam(msg) else "ham"
    print(f"ML: {verdict} ({p_spam:5.1%} spam) | rules: {rule:<4} | {msg[:70]}")

ML: ham  (26.9% spam) | rules: spam | Bhai, free ho toh call kar, notes chahiye for tomorrow's exam!
ML: SPAM (77.4% spam) | rules: ham  | Dear customer, your KYC is pending. Update now at kyc-update-bank.in o
ML: ham  ( 0.6% spam) | rules: ham  | Congrats! U hav w0n a 1Ph0ne. Cl!ck bit.ly/xyz
ML: ham  ( 0.0% spam) | rules: ham  | Are we meeting at the canteen after the lab?
